# 02 — Bronze: Ingestão CSV → Delta Lake (Spark SQL)

Lê os 11 CSVs e grava como tabelas Delta.

**Técnica Spark SQL:** `spark.read.csv()` → `createOrReplaceTempView()` → `INSERT INTO ... SELECT *, current_timestamp()`.

Full Load — DELETE + INSERT a cada execução.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR, DATA_DIR

spark = get_spark("NorthwindDW SQL - 02 Bronze Ingest")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 00:28:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/30 00:28:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 11}


In [3]:
def ingest_csv(csv_name, table_name, cast_cols=None, multiline=False):
    """Lê CSV → temp view → DELETE + INSERT com current_timestamp().
    Full Load: equivalente ao overwrite do DataFrame API."""
    from pyspark.sql import functions as F
    df = (spark.read.option("header", True).option("inferSchema", True)
          .option("multiLine", multiline).csv(f"{DATA_DIR}/{csv_name}"))
    if cast_cols:
        for col_name, col_type in cast_cols.items():
            df = df.withColumn(col_name, F.col(col_name).cast(col_type))
    view = table_name.replace(".", "_") + "_raw"
    df.createOrReplaceTempView(view)
    spark.sql(f"DELETE FROM {table_name}")
    col_list = ", ".join(df.columns)
    spark.sql(f"INSERT INTO {table_name} ({col_list}, _LoadTimestamp)"
              f" SELECT {col_list}, current_timestamp() FROM {view}")
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {table_name}").collect()[0]["n"]
    print(f"  {table_name}: {n} linhas")
    return n

In [4]:
print("Ingestão bronze — tabelas de referência:")
ingest_csv("customers.csv", "bronze.customers")
ingest_csv("employees.csv", "bronze.employees",
    cast_cols={"BirthDate": "timestamp", "HireDate": "timestamp"}, multiline=True)
ingest_csv("products.csv", "bronze.products",
    cast_cols={"UnitPrice": "double", "UnitsInStock": "int",
               "UnitsOnOrder": "int", "ReorderLevel": "int", "Discontinued": "boolean"})
ingest_csv("categories.csv", "bronze.categories")
ingest_csv("suppliers.csv", "bronze.suppliers")
ingest_csv("shippers.csv", "bronze.shippers")
print("OK")

Ingestão bronze — tabelas de referência:
  bronze.customers: 91 linhas
  bronze.employees: 9 linhas
  bronze.products: 77 linhas
  bronze.categories: 8 linhas
  bronze.suppliers: 29 linhas
  bronze.shippers: 3 linhas
OK


In [5]:
print("Ingestão bronze — tabelas transacionais:")
ingest_csv("orders.csv", "bronze.orders",
    cast_cols={"OrderDate": "timestamp", "RequiredDate": "timestamp",
               "ShippedDate": "timestamp", "Freight": "double"})
ingest_csv("order_details.csv", "bronze.order_details",
    cast_cols={"UnitPrice": "double", "Discount": "double"})
print("OK")

Ingestão bronze — tabelas transacionais:
  bronze.orders: 830 linhas
  bronze.order_details: 2155 linhas
OK


In [6]:
print("Ingestão bronze — territórios:")
ingest_csv("territories.csv", "bronze.territories")
ingest_csv("region.csv", "bronze.region")
ingest_csv("employeeterritories.csv", "bronze.employee_territories")
print("OK")

Ingestão bronze — territórios:
  bronze.territories: 53 linhas
  bronze.region: 4 linhas
  bronze.employee_territories: 49 linhas
OK


In [7]:
bronze_tables = [
    "bronze.customers", "bronze.employees", "bronze.products",
    "bronze.categories", "bronze.suppliers", "bronze.shippers",
    "bronze.orders", "bronze.order_details",
    "bronze.territories", "bronze.region", "bronze.employee_territories",
]
print("\nContagens bronze:")
for t in bronze_tables:
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {t}").collect()[0]["n"]
    print(f"  {t:<40} {n:>5} linhas")
print("\nEsperado: customers=91, employees=9, products=77, categories=8")
print("          suppliers=29, shippers=3, orders=830, order_details=2155")
print("          territories=53, region=4, employee_territories=49")


Contagens bronze:
  bronze.customers                            91 linhas
  bronze.employees                             9 linhas
  bronze.products                             77 linhas
  bronze.categories                            8 linhas
  bronze.suppliers                            29 linhas
  bronze.shippers                              3 linhas
  bronze.orders                              830 linhas
  bronze.order_details                      2155 linhas
  bronze.territories                          53 linhas
  bronze.region                                4 linhas
  bronze.employee_territories                 49 linhas

Esperado: customers=91, employees=9, products=77, categories=8
          suppliers=29, shippers=3, orders=830, order_details=2155
          territories=53, region=4, employee_territories=49


In [8]:
spark.sql("""
    SELECT OrderID, CustomerID, EmployeeID,
           OrderDate, RequiredDate, ShippedDate, ShipCountry
    FROM bronze.orders LIMIT 5
""").show(truncate=False)

+-------+----------+----------+-------------------+-------------------+-------------------+-----------+
|OrderID|CustomerID|EmployeeID|OrderDate          |RequiredDate       |ShippedDate        |ShipCountry|
+-------+----------+----------+-------------------+-------------------+-------------------+-----------+
|10248  |VINET     |5         |1996-07-04 00:00:00|1996-08-01 00:00:00|1996-07-16 00:00:00|France     |
|10249  |TOMSP     |6         |1996-07-05 00:00:00|1996-08-16 00:00:00|1996-07-10 00:00:00|Germany    |
|10250  |HANAR     |4         |1996-07-08 00:00:00|1996-08-05 00:00:00|1996-07-12 00:00:00|Brazil     |
|10251  |VICTE     |3         |1996-07-08 00:00:00|1996-08-05 00:00:00|1996-07-15 00:00:00|France     |
|10252  |SUPRD     |4         |1996-07-09 00:00:00|1996-08-06 00:00:00|1996-07-11 00:00:00|Belgium    |
+-------+----------+----------+-------------------+-------------------+-------------------+-----------+

